# Ticha HTR — label smoothing and the cosine schedule, one change at a time

In `ticha_trocr_finetuning.ipynb` the first run (linear schedule, no label smoothing, 5 epochs) and the second (label smoothing 0.1, cosine schedule, up to 12 epochs) differ in three things at once, so the second run's gain (validation CER 18.7% → 17.9%) can't be split between them. This notebook trains the missing runs with **everything else identical to the second run**:

| Run | Label smoothing | Schedule | Epochs | Status |
|---|---|---|---|---|
| A | none | linear | up to 12 | trained here |
| B | 0.1 | linear | up to 12 | trained here |
| C | none | cosine | up to 12 | trained here |
| D | 0.1 | cosine | up to 12 | `trocr-base-handwritten-finetuned-smooth-cosine` (existing) |
| first run | none | linear | 5 | `trocr-base-handwritten-finetuned` (existing, for reference) |
| E | 0.1 | linear | up to 12, **with augmentation** | trained here (added after B beat D on validation) |
| D + augmentation | 0.1 | cosine | up to 12, with augmentation | `trocr-base-handwritten-finetuned-smooth-cosine-aug` (existing, `e2e-aug`) |

- **A vs D** shows what smoothing and cosine together add once the epochs are equal; **A vs the first run** shows what the longer training alone adds. **B** and **C** split smoothing from the schedule.
- **Same everything else:** the model setup, collator and arguments are those of the "Experiment: label smoothing, cosine schedule and augmentation" section of `ticha_trocr_finetuning.ipynb` (the shared pieces are in `trocr_model.py`) (AdamW, lr 5e-5, weight decay 0.01, 5% warmup, batch 8, seed 42, bf16), with early stopping on validation CER (patience 2) and the best epoch kept.
- **Same data for every run:** the original train lines of dataset **v1.1** (5,305 lines after cleaning), the lines D and the first run were trained on, loaded and cleaned by `trocr_data.py`.
- **E vs D + augmentation** asks whether B's edge over D (validation 17.4% vs 17.9%) survives augmentation. E uses the augmented train lines exactly as `e2e-aug` did (v1.1, cleaned like the originals, original + augmented shuffled with seed 42, 26,525 lines), so the schedule is the only difference. E also scores a fixed sample of 710 original train lines every epoch (as many as validation), so its training CER per epoch sits next to its validation CER. Both are also scored on the degraded test copy (`test_degraded`, the same lines).
- **Same evaluation for all runs:** validation and test decoded with 2 beams (as during training) and start token `0`, and a paired bootstrap over whole documents against D.

**Before running:** train the three runs of [`ticha_trocr_finetuning.ipynb`](ticha_trocr_finetuning.ipynb) (D, the first run and D + augmentation); this notebook reads them from `$TICHA_RUNS_DIR` (default `~/ticha-runs`).

Run on the VM (kernel **Python (ticha-trocr)**): about 1 hour per run. A trained run is reloaded on a rerun; its epoch checkpoints are deleted once its best model is saved (the disk has room for one run's checkpoints at a time).

In [ ]:
import sys
from pathlib import Path

# The kernel may start in the repository root or in notebooks/train: find the shared modules
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks' / 'train' / 'trocr_setup.py').exists())
sys.path[:0] = [str(ROOT / 'notebooks' / 'train'), str(ROOT / 'src')]

import gc
import json
import os
import re
import shutil
import time

import pandas as pd
import torch
from IPython.display import display
from transformers import (EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments, TrOCRProcessor,
                          VisionEncoderDecoderModel)

from trocr_data import fixed_sample, load_clean_splits, load_test_degraded, load_train_aug
from trocr_eval import decode_split, evaluate, paired_bootstrap
from trocr_model import (TrOCRSmoothingCollator, cer_wer_metrics, configure_for_decoding, max_target_length,
                         pretrained_model)
from trocr_setup import MODEL_ID, RUNS_DIR, SEED, gpu_memory_gb, require_gpu

BASE = RUNS_DIR / 'trocr-base-handwritten-finetuned'
OUT = RUNS_DIR / 'ablation-smoothing-schedule'
SMOKE = os.environ.get('TICHA_SMOKE') == '1'  # quick end-to-end check in a scratch folder
if SMOKE:
    OUT = RUNS_DIR / 'smoke-ablation'
OUT.mkdir(parents=True, exist_ok=True)

ABLATION_RUNS = {
    'A no smoothing, linear': dict(label_smoothing_factor=0.0, lr_scheduler_type='linear'),
    'B smoothing, linear': dict(label_smoothing_factor=0.1, lr_scheduler_type='linear'),
    'C no smoothing, cosine': dict(label_smoothing_factor=0.0, lr_scheduler_type='cosine'),
    'E smoothing, linear + augmentation': dict(label_smoothing_factor=0.1, lr_scheduler_type='linear', augmented=True),
}
EXISTING_RUNS = {
    'D smoothing, cosine (existing)': RUNS_DIR / 'trocr-base-handwritten-finetuned-smooth-cosine',
    'first run: linear, 5 epochs (existing)': BASE,
    'D + augmentation (existing)': RUNS_DIR / 'trocr-base-handwritten-finetuned-smooth-cosine-aug',
}
AUGMENTED_RUNS = ['E smoothing, linear + augmentation', 'D + augmentation (existing)']
if SMOKE:  # disk is tight: the smoke check trains only the new run
    ABLATION_RUNS = {k: v for k, v in ABLATION_RUNS.items() if k.startswith('E ')}
EVAL_BEAMS = 2
DECODE_BATCH = 16

gpu = require_gpu('run this notebook on the VM')
USE_BF16 = torch.cuda.is_bf16_supported()
BATCH_SIZE = 8 if gpu_memory_gb() >= 20 else 2
print('GPU:', gpu, '| batch size:', BATCH_SIZE, '| bf16:', USE_BF16, '| outputs:', OUT)

## Data

Loaded and cleaned by `trocr_data.py`, the same lines as in the fine-tuning notebook: uncertain readings kept as text in train, dropped from validation and test. Run E also needs the augmented train lines and the degraded copy of the test lines. The `assert`s stop the notebook if the lines differ from those D and the first run were trained on.

In [ ]:
ds = load_clean_splits()
assert SMOKE or len(ds['train']) == 5305, 'not the train set D and the first run were trained on'
train_aug = load_train_aug(ds['train'])
assert SMOKE or len(train_aug) == 26525, 'not the train set e2e-aug was trained on'
ds['test_degraded'] = load_test_degraded(ds['test'])
# Run E's training CER per epoch: a fixed sample of original (not augmented) train lines, as many as validation.
train_sample = fixed_sample(ds['train'], len(ds['validation']))

if SMOKE:
    ds['train'], ds['validation'], ds['test'] = (ds['train'].select(range(64)), ds['validation'].select(range(16)),
                                                 ds['test'].select(range(16)))
    ds['test_degraded'], train_aug = ds['test_degraded'].select(range(16)), train_aug.select(range(64))
    train_sample = ds['train'].select(range(16))
meta = {name: ds[name].remove_columns('image').to_pandas() for name in ['validation', 'test', 'test_degraded']}

## Model setup and training (copied from the experiment section of `ticha_trocr_finetuning.ipynb`)

In [ ]:
processor = TrOCRProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
MAX_TARGET_LENGTH, _, _ = max_target_length(tokenizer, ds['train']['text'])
print('MAX_TARGET_LENGTH:', MAX_TARGET_LENGTH)
compute_metrics = cer_wer_metrics(processor)  # validation CER and WER


def run_dir(name):
    return OUT / re.sub(r'[^A-Za-z0-9]+', '-', name).strip('-').lower()


for name, changes in ABLATION_RUNS.items():
    changes = dict(changes)
    augmented = changes.pop('augmented', False)
    train_set = train_aug if augmented else ds['train']
    # E also scores the train sample each epoch; with two eval sets the metrics are named eval_validation_cer and
    # eval_train_sample_cer, and the best epoch is still chosen on validation.
    eval_set = {'validation': ds['validation'], 'train_sample': train_sample} if augmented else ds['validation']
    best_metric = 'validation_cer' if augmented else 'cer'
    out_dir = run_dir(name)
    if (out_dir / 'model.safetensors').exists():
        print(f'{name}: using the saved model in {out_dir}')
        continue
    args = Seq2SeqTrainingArguments(
        output_dir=str(out_dir), per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=8 // BATCH_SIZE, optim='adamw_torch', learning_rate=5e-5, weight_decay=0.01,
        warmup_ratio=0.05, num_train_epochs=12, eval_strategy='epoch', save_strategy='epoch',
        logging_strategy='steps', logging_steps=100, save_total_limit=2, load_best_model_at_end=True,
        metric_for_best_model=best_metric, greater_is_better=False, predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH, generation_num_beams=2, fp16=not USE_BF16, bf16=USE_BF16,
        remove_unused_columns=False, dataloader_num_workers=4, report_to='none', seed=SEED,
        **changes, **({'max_steps': 10} if SMOKE else {}))
    # Every run starts from the pretrained checkpoint, decodes validation with 2 beams, and uses the smoothing
    # collator (it adds the decoder inputs), so all runs see the same batches.
    model = pretrained_model(MODEL_ID, tokenizer, MAX_TARGET_LENGTH, num_beams=2)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_set, eval_dataset=eval_set,
                             data_collator=TrOCRSmoothingCollator(processor, MAX_TARGET_LENGTH),
                             compute_metrics=compute_metrics,
                             callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    print(f'\n=== {name}: {changes}, {len(train_set):,} training lines -> {out_dir}', flush=True)
    t0 = time.time()
    trainer.train()
    trainer.save_model(str(out_dir))  # the best checkpoint (load_best_model_at_end)
    processor.save_pretrained(str(out_dir))
    trainer.save_state()
    print(f'{name}: trained in {(time.time() - t0) / 60:.0f} min', flush=True)
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    for checkpoint in out_dir.glob('checkpoint-*'):  # the best model is saved above; keep room for the next run
        shutil.rmtree(checkpoint)

In [ ]:
def training_summary(model_dir):
    state = json.loads((Path(model_dir) / 'trainer_state.json').read_text())
    key = 'eval_cer' if any('eval_cer' in h for h in state['log_history']) else 'eval_validation_cer'
    evals = [h for h in state['log_history'] if key in h]
    best = min(evals, key=lambda h: h[key])
    return {'best epoch': int(round(best['epoch'])), 'epochs run': int(round(evals[-1]['epoch'])),
            'validation CER (training log)': best[key], 'per epoch': [round(h[key], 4) for h in evals]}


all_runs = {name: run_dir(name) for name in ABLATION_RUNS}
all_runs.update(EXISTING_RUNS)
curves = pd.DataFrame({name: training_summary(path) for name, path in all_runs.items()}).T
print('Validation CER per epoch, from each run\'s training log (2 beams):')
display(curves)


# Run E: training CER (710 original train lines) next to validation CER, per epoch.
e_state = json.loads((run_dir('E smoothing, linear + augmentation') / 'trainer_state.json').read_text())
e_curve = {}
for h in e_state['log_history']:
    for split in ['validation', 'train_sample']:
        if f'eval_{split}_cer' in h:
            e_curve.setdefault(int(round(h['epoch'])), {})[f'{split} CER'] = h[f'eval_{split}_cer']
e_curve = pd.DataFrame(e_curve).T.rename_axis('epoch')
print('Run E, CER per epoch (2 beams): training sample vs validation')
display(e_curve.round(4))
e_curve.to_csv(OUT / 'e_train_vs_validation_cer.csv')

## The same evaluation for all seven runs

Validation and test decoded with 2 beams and start token `0`. Cached predictions are scored only on the lines of the loaded splits. `evaluate` and `paired_bootstrap` are in `trocr_eval.py`.

In [ ]:
references = {s: dict(zip(meta[s]['id'], meta[s]['text'])) for s in meta}
languages = {s: dict(zip(meta[s]['id'], meta[s]['language'])) for s in meta}
# Score only the lines of the loaded splits (a cached prediction file can cover more lines).
keep = {split: set(ids).__contains__ for split, ids in references.items()}


@torch.inference_mode()
def predict(model_dir, split_name):
    """{line id: text} of a split, cached in OUT/predictions."""
    path = OUT / 'predictions' / f"{re.sub(r'[^A-Za-z0-9]+', '-', str(model_dir.name)).strip('-')}__{split_name}.csv"
    if path.exists():
        df = pd.read_csv(path, keep_default_na=False)
        return dict(zip(df['id'], df['prediction']))
    model = configure_for_decoding(VisionEncoderDecoderModel.from_pretrained(model_dir), tokenizer, MAX_TARGET_LENGTH)
    model = model.to('cuda').eval()
    out = decode_split(model, processor, ds[split_name], num_beams=EVAL_BEAMS, max_length=MAX_TARGET_LENGTH,
                       batch_size=DECODE_BATCH, use_bf16=USE_BF16)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({'id': list(out), 'reference': [references[split_name][i] for i in out],
                  'prediction': list(out.values())}).to_csv(path, index=False)
    del model
    torch.cuda.empty_cache()
    return out


def scores(split, preds):
    """CER and WER of a split; for test also the number of lines and the CER by language."""
    result = evaluate(references[split], preds, languages[split], keep=keep[split])
    row = {f'{split} CER': result['CER'], f'{split} WER': result['WER']}
    if split == 'test':
        row['test lines'] = result['lines']
        row.update({f'test {k}': v for k, v in result.items() if k.startswith('CER ')})
    return row


def compare(base, new):
    """CER difference new - base on test, with its 95% interval (paired bootstrap over documents)."""
    return paired_bootstrap(references['test'], base, new, keep=keep['test'], seed=SEED)


predictions = {name: {s: predict(Path(path), s) for s in ['validation', 'test']} for name, path in all_runs.items()}
table = pd.DataFrame({name: {**{k: v for k, v in training_summary(path).items() if k != 'per epoch'},
                             **scores('validation', predictions[name]['validation']),
                             **scores('test', predictions[name]['test'])}
                      for name, path in all_runs.items()}).T
print('All runs, decoded the same way (2 beams, start token 0)')
display(table)
table.to_csv(OUT / 'ablation_results.csv')

reference_run = 'D smoothing, cosine (existing)'
for name in all_runs:
    if name != reference_run:
        r = compare(predictions[reference_run]['test'], predictions[name]['test'])
        print(f"{name} minus D, test CER: {r['difference']:+.4f} [{r['low']:+.4f}, {r['high']:+.4f}], "
              f"better than D in {r['share better']:.1%} of resamples")
if 'A no smoothing, linear' in predictions:  # not in the smoke check, which trains only E
    r = compare(predictions['first run: linear, 5 epochs (existing)']['test'], predictions['A no smoothing, linear']['test'])
    print(f"A minus the first run (longer training only), test CER: {r['difference']:+.4f} "
          f"[{r['low']:+.4f}, {r['high']:+.4f}], better in {r['share better']:.1%} of resamples")

# With augmentation: does the linear schedule (E) still beat cosine (D + augmentation)?
e_name, daug_name = AUGMENTED_RUNS
r = compare(predictions[daug_name]['test'], predictions[e_name]['test'])
print(f"E minus D + augmentation, test CER: {r['difference']:+.4f} [{r['low']:+.4f}, {r['high']:+.4f}], "
      f"better in {r['share better']:.1%} of resamples")
degraded_table = pd.DataFrame({name: {**scores('test', predictions[name]['test']),
                                      **scores('test_degraded', predict(Path(all_runs[name]), 'test_degraded'))}
                               for name in AUGMENTED_RUNS}).T[['test CER', 'test_degraded CER', 'test WER', 'test_degraded WER']]
degraded_table['degradation (CER points)'] = 100 * (degraded_table['test_degraded CER'] - degraded_table['test CER'])
print('Real vs degraded scans:')
display(degraded_table)
degraded_table.to_csv(OUT / 'augmented_runs_degraded.csv')

## How to read the results

- **First run → A:** the effect of training up to 12 epochs with early stopping, with the first run's settings.
- **A → D:** what label smoothing and the cosine schedule add together, at equal epochs. **B** and **C** show each alone.
- **E vs D + augmentation:** the schedule with augmentation. Only a validation gain counts for choosing; test and its interval confirm it.
- These are single runs with one seed. Differences of a few tenths of a point may come from randomness; a difference only counts if its bootstrap interval excludes zero, and even then a second seed is the real check.